# Viseven: content-engagement reports that clients can trust
Independent case study based on public data · not affiliated with or endorsed by Viseven.

**PUBLIC DATA**: Eurostat, European Medicines Agency, ClinicalTrials.gov, npm, public holidays. **SIMULATED**: the fictional pharma company *Client P* (8 markets, Dec 2025 - Nov 2026), its HCPs (identifiers only), contents and the local REST API. This notebook only reads the outputs built by the scripts (see README).

In [1]:
import duckdb, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
con = duckdb.connect(str(ROOT / 'data' / 'warehouse.duckdb'), read_only=True)
q = lambda s: con.execute(s).df()
out = lambda n: pd.read_csv(ROOT / 'data' / 'output' / f'{n}.csv')

## 1. The real market (PUBLIC DATA)
Practising physicians (Eurostat, 2024), recruiting trials by market (ClinicalTrials.gov REST load, `totalCount` checked), EU medicines still authorised by year (Source: European Medicines Agency; survivor bias).

In [2]:
out('physicians')

,market,year,physicians,per_100k,client_p_hcps_simulated
0,DE,2024,397042,475.40,1538
1,IT,2024,339256,575.43,1313
2,FR,2024,270874,393.85,1048
3,ES,2024,226667,463.78,877
4,PL,2024,141193,386.20,546
5,RO,2024,72647,381.24,281
6,BG,2024,30086,467.07,116


In [3]:
out('trials_by_market')[['market', 'recruiting_studies', 'oncology_pct', 'api_total_count', 'received', 'complete']]

,market,recruiting_studies,oncology_pct,api_total_count,received,complete
0,FR,6074,25.1,6074,6074,True
1,IT,4096,32.3,4096,4096,True
2,ES,3121,33.6,3121,3121,True
3,DE,2890,29.6,2890,2890,True
4,PL,1390,28.8,1390,1390,True
5,RO,411,34.5,411,411,True
6,BG,285,17.9,285,285,True
7,UA,147,29.9,147,147,True


In [4]:
out('ema_by_year_area').query('year >= 2022').pivot_table(index='year', columns='area_group', values='medicines', fill_value=0)

area_group,Cardiovascular & metabolic,Immunology & inflammation,Infectious diseases,Neurology & psychiatry,Oncology,Other,Rare & genetic
year,,,,,,,
2022,10.0,5.0,6.0,10.0,21.0,28.0,4.0
2023,7.0,5.0,5.0,9.0,18.0,18.0,3.0
2024,8.0,12.0,8.0,5.0,29.0,26.0,6.0
2025,13.0,7.0,8.0,8.0,33.0,33.0,6.0
2026,13.0,9.0,7.0,3.0,12.0,22.0,8.0


## 2. The API and the loads (SIMULATED)
Full load to 01/09/2026, then the 6-hourly incremental job replayed as it ran in production (v1), then a backfill. Every run is logged.

In [5]:
q('SELECT * FROM out_load_runs')

,job,mode,runs,failed_runs,requests,http_5xx,http_429,rows_received,rows_inserted,duplicates_ignored,seconds
0,full,full,1,0,63.0,0.0,0.0,307042.0,307042.0,0.0,15.6
1,v1,incremental,341,1,472.0,5.0,0.0,114123.0,110909.0,3214.0,120.3
2,backfill,backfill,1,0,3.0,0.0,0.0,715.0,715.0,0.0,0.7


In [6]:
q("SELECT run_id, as_of, updated_since, status, watermark_after FROM load_runs WHERE status LIKE 'failed%'")

,run_id,as_of,updated_since,status,watermark_after
0,55,2026-09-14T09:00:00Z,2026-09-14T03:00:00Z,failed: /v1/events failed after 5 attempts,2026-09-14T09:00:00Z


In [7]:
out('benchmark_load')[['scenario', 'requests', 'seconds', 'rows_loaded', 'lost_events']]

,scenario,requests,seconds,rows_loaded,lost_events
0,full_500,854,204.1,418666,0
1,full_5000,102,37.9,418666,0
2,incremental_day,11,3.1,1697,0
3,incident_v1,19,13.1,1088,715
4,incident_v2,24,15.5,1803,0


## 3. Verification
Checks C1-C8 before and after the backfill (`src/verify.py`).

In [8]:
v = out('verification_checks')
v.pivot(index='check', columns='run_label', values='status')[['before_backfill', 'after_backfill']]

run_label,before_backfill,after_backfill
check,,
C1,FAIL,PASS
C2,PASS,PASS
C3,PASS,PASS
C4,PASS,PASS
C5,WARN,WARN
C6,WARN,WARN
C7,ALERT,ALERT
C8,PASS,PASS


In [9]:
out('reconciliation_incident')

,run_label,day,source_events,warehouse_events,gap
0,before_backfill,2026-09-12,55,55,0
1,before_backfill,2026-09-13,44,44,0
2,before_backfill,2026-09-14,1801,1086,-715
3,before_backfill,2026-09-15,1895,1895,0
4,before_backfill,2026-09-16,1916,1916,0
5,after_backfill,2026-09-12,55,55,0
6,after_backfill,2026-09-13,44,44,0
7,after_backfill,2026-09-14,1801,1801,0
8,after_backfill,2026-09-15,1895,1895,0
9,after_backfill,2026-09-16,1916,1916,0


In [10]:
out('timezone_check')

,market,sessions,end_before_start_as_written,end_before_start_after_fix,naive_timestamps
0,BG,737,0,0,0
1,DE,9712,0,0,0
2,ES,5699,0,0,0
3,FR,6625,0,0,0
4,IT,8408,0,0,0
5,PL,3556,0,0,0
6,RO,1744,304,0,2062
7,UA,1751,0,0,0


## 4. KPIs
Engagement time capped at the last slide + 30 s; CTR by segment (Simpson); point-in-time content status.

In [11]:
out('engagement_definition')

,channel,sessions,idle_sessions,naive_min_per_session,capped_min_per_session,naive_inflation
0,e-detail,35097,617,3.21,2.32,0.3799
1,web,3777,0,0.80,0.80,0.0000


In [12]:
out('simpson').pivot(index='segment', columns='market', values='ctr')

market,DE,PL
segment,,
All,0.0400,0.0534
GP,0.0725,0.0624
Pharmacist,0.0597,0.0505
Specialist,0.0304,0.0270


In [13]:
out('simpson_standardised')

,market,ctr_standardised,ctr_raw
0,DE,0.0497,0.0400
1,PL,0.0431,0.0534


In [14]:
out('point_in_time_restatement')

,published_on,march_sessions_current_status_logic,march_sessions_point_in_time
0,2026-04-01,3746,3775
1,2026-07-01,3156,3775
2,2026-10-01,3043,3775
3,2026-11-25,2360,3775


## 5. The monthly report (October 2026)
Same values as the Excel workbook (test T14).

In [15]:
out('kpi_market').query("month == '2026-10'")

,month,market,hcps_targeted,hcps_reached,coverage,frequency,edetail_sessions,engaged_min_per_session,naive_min_per_session,emails_sent,open_rate,ctr,webinar_joins
80,2026-10,BG,116,73,0.6293,2.151,66,2.090,3.098,167,0.2814,0.0479,36.0
81,2026-10,DE,1538,805,0.5234,2.123,884,2.257,2.730,2205,0.2975,0.0367,88.0
82,2026-10,ES,877,434,0.4949,2.055,490,2.299,3.066,1254,0.2799,0.0407,NaN
83,2026-10,FR,1048,558,0.5324,2.072,608,2.289,3.247,1541,0.2927,0.0344,44.0
84,2026-10,IT,1313,691,0.5263,1.957,616,2.275,3.495,1881,0.3083,0.0372,86.0
85,2026-10,PL,546,287,0.5256,2.265,321,2.338,3.242,876,0.3208,0.0548,NaN
86,2026-10,RO,281,143,0.5089,2.126,157,2.395,3.865,404,0.3069,0.0569,NaN
87,2026-10,UA,281,139,0.4947,2.151,156,2.330,3.608,421,0.2969,0.0428,NaN


In [16]:
pd.read_csv(ROOT / 'data' / 'output' / 'workbook_check.csv').assign(diff=lambda d: d.excel - d.python).describe().loc[['count', 'max']]

,excel,python,diff
count,32.0,32.0,32.0
max,884.0,884.0,0.0


## 6. Incident and compliance
Outage of 14/09/2026 (v1 vs v2), sessions after MLR expiry, and the August dip read against working days.

In [17]:
out('benchmark_load').query("scenario.str.startswith('incident')")[['scenario', 'requests', 'rows_loaded', 'lost_events']]

,scenario,requests,rows_loaded,lost_events
3,incident_v1,19,1088,715
4,incident_v2,24,1803,0


In [18]:
out('compliance')[['content_id', 'market', 'brand', 'mlr_expiry', 'sessions', 'hcps', 'max_days_after_expiry']]

,content_id,market,brand,mlr_expiry,sessions,hcps,max_days_after_expiry
0,CNT006,IT,Brand B,2026-10-21,230,199,19
1,CNT014,IT,Brand B,2026-11-03,182,160,19


In [19]:
out('august_check')

,year,july,august,change,change_per_working_day
0,2025,3177,2099,-0.3393,-0.2623
1,2026,3167,2185,-0.3101,-0.2481


## 7. Tests and improvements
All assertions of `sql/99_tests.sql`. Proposals with before / after: `docs/report_improvements.md`.

In [20]:
q('SELECT * FROM test_results ORDER BY test')

,test,result,detail
0,T01 event_id unique after de-duplication,PASS,418666 events
1,T02 report totals = sum of markets,PASS,12 months compared
2,T03 e-detail sessions: KPI table = report cells,PASS,34685 sessions
3,T04 no published group of 1-4 HCPs,PASS,117 cells suppressed (primary + secondary)
4,T05 E1 outage: gap found then closed by backfill,PASS,"-715 events before backfill, 0 after"
5,T06 E2 duplicates received and ignored,PASS,3214 duplicate rows
6,T07 E3 RO time zone found and fixed,PASS,"304 sessions ending before start as written, 0..."
7,"T08 E4 Simpson: DE lower overall, higher in ev...",PASS,overall DE 0.04 vs PL 0.0534
8,T09 E5 idle sessions inflate naive engagement ...,PASS,+38.0% naive vs capped
9,T10 E6 current-status filter rewrites past months,PASS,March 2026: 3746 -> 2360 sessions (point-in-ti...
